# Day 14: Schema Handling

**Goal:** Define schemas explicitly, and see how Spark reacts when data breaks them.

**Key pattern:** PERMISSIVE keeps bad rows as null, DROPMALFORMED silently removes them, FAILFAST stops the read entirely.

In [0]:

#Write a sample CSV into your Volume (reusing the Day 12/13 path)
volume_path = "/Volumes/workspace/default/day12_files"

csv_content = """id,name,department,salary
1,Alice,IT,55000
2,Ben,HR,48000
3,Cara,IT,N/A
"""
with open(f"{volume_path}/day14_sample.csv", "w") as f:
    f.write(csv_content)

In [0]:
#. Define an explicit schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
])

In [0]:
#Read with the explicit schema and check what happened to the bad value

df = spark.read.schema(schema).option("header", True).csv(f"{volume_path}/day14_sample.csv")
df.display()
df.printSchema()

In [0]:
# Try Fail Fast and watch it error instead
df_strict = spark.read.schema(schema).option("header", True).option("mode", "FAILFAST").csv(f"{volume_path}/day14_sample.csv")
df_strict.display()


## Note: found a bug bigger than expected

Assumed a missing column would just become null. Actually discovered:
providing an explicit schema + header=True matches columns by POSITION,
not name - a missing middle column shifts every column after it into
the wrong field. Verify column order matches the schema, not just presence.

In [0]:
# Try the missing column scenario
csv_missing_col = """id,name,salary
1,Alice,55000
2,Ben,48000
"""
with open(f"{volume_path}/day14_missing_col.csv", "w") as f:
    f.write(csv_missing_col)

df_missing = spark.read.schema(schema).option("header", True).csv(f"{volume_path}/day14_missing_col.csv")
df_missing.display()

In [0]:
# Try DROPMALFORMED
df_drop = spark.read.schema(schema).option("header", True).option("mode", "DROPMALFORMED").csv(f"{volume_path}/day14_sample.csv")
df_drop.display()
print("Row count with DROPMALFORMED:", df_drop.count())

In [0]:
#Explicitly catch the corrupt record instead of just seeing a null (Scenario C)

# Note Check: this should actually show you which row was malformed, rather than just a silent null — the difference between Scenario B and Scenario C from the notes.

schema_with_corrupt = schema.add(StructField("_corrupt_record", StringType(), True))

df_corrupt = (spark.read
    .schema(schema_with_corrupt)
    .option("header", True)
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv(f"{volume_path}/day14_sample.csv")
)
df_corrupt.display()
df_corrupt.filter(df_corrupt["_corrupt_record"].isNotNull()).display()

In [0]:
#Nested Schema Example
from pyspark.sql.types import ArrayType

schema_nested = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("skills", ArrayType(StringType()), True),
])

data_nested = [(1, "Alice", ["Python", "SQL", "PySpark"]), (2, "Ben", ["Excel", "SQL"])]
df_nested = spark.createDataFrame(data_nested, schema_nested)
df_nested.display()
df_nested.printSchema()

In [0]:
# Build the same dataset three ways, side by side

df_no_schema = spark.read.option("header", True).csv(f"{volume_path}/day14_sample.csv")
df_no_schema.printSchema()   # everything a string

df_infer = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/day14_sample.csv")
df_infer.printSchema()       # inferSchema's guess

df.printSchema()             # your explicit schema from Step 3

In [0]:
# ============================================================
# Real-world file-read checklist - the 4 steps, every new file
# ============================================================

# STEP 1: Look first, don't assume. inferSchema shows you Spark's best guess.
df = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/day14_sample.csv")
df.printSchema()

# STEP 2: Go find the actual bad value(s) - don't just force a fix blindly.
df.filter(~df.salary.rlike("^[0-9.]+$")).select("salary").distinct().display()

# STEP 3: Now define the real schema on purpose.
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
])
df = spark.read.schema(schema).option("header", True).csv(f"{volume_path}/day14_sample.csv")

# STEP 4: Never trust a read - verify it. Check every column for nulls.
for field in schema.fields:
    n = df.filter(df[field.name].isNull()).count()
    print(field.name, ":", n, "nulls")

# STEP 5: A print() disappears once the notebook stops running.
# Record the result somewhere persistent instead, so it survives
# past this one session - this is the seed of Day 47's audit logs.
from datetime import datetime

results = []
for field in schema.fields:
    n = df.filter(df[field.name].isNull()).count()
    results.append((datetime.now(), "day14_sample.csv", field.name, n))

log_df = spark.createDataFrame(results, ["checked_at", "source_file", "column_name", "null_count"])
log_df.write.format("delta").mode("append").saveAsTable("workspace.default.dq_check_log")

# THE RULE: never trust a file read until you've printed its schema,
# checked every column for nulls, AND recorded that check somewhere
# persistent - not just printed to a notebook that will eventually close.